In [31]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REQUIRED_DATA_FILES = {
    "train_demos.pkl",
    "valid_scenarios.pkl",
    "test_scenarios.pkl",
}


def contains_required_files(folder: Path) -> bool:
    return folder.is_dir() and REQUIRED_DATA_FILES.issubset(
        {path.name for path in folder.iterdir() if path.is_file()}
    )


def find_data_directory() -> Path | None:
    candidates = [
        Path("/kaggle/input/competitions/ioai-2026-home-task-2"),
        Path("/kaggle/input/ioai-2026-home-task-2"),
        Path("/content/data"),
        Path("data"),
    ]

    for candidate in candidates:
        if contains_required_files(candidate):
            return candidate

    # Kaggle puede montar la competencia bajo una carpeta con nombre ligeramente distinto.
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        for train_file in kaggle_root.rglob("train_demos.pkl"):
            if contains_required_files(train_file.parent):
                return train_file.parent

    return None


DATA_DIR = find_data_directory()

if DATA_DIR is None:
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])

    import gdown

    download_dir = Path("data")
    download_dir.mkdir(parents=True, exist_ok=True)

    print("No se encontraron los datos localmente. Descargando la carpeta pública...")
    gdown.download_folder(
        id="1DXFDoY9bqulMBFacyDShVIx8Sa7Z5Wpa",
        output=str(download_dir),
        quiet=False,
        use_cookies=False,
    )
    DATA_DIR = find_data_directory()

if DATA_DIR is None:
    raise FileNotFoundError(
        "No fue posible encontrar train_demos.pkl, valid_scenarios.pkl y "
        "test_scenarios.pkl. En Kaggle, agregue los datos de la competencia "
        "al notebook; en Colab, habilite internet para la descarga pública."
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("Archivos:", sorted(path.name for path in DATA_DIR.iterdir()))

DATA_DIR: /kaggle/input/competitions/ioai-2026-home-task-2
OUTPUT_DIR: /kaggle/working
Archivos: ['test_scenarios.pkl', 'train_demos.pkl', 'valid_scenarios.pkl']


In [32]:
import json
import pickle
import random
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from IPython.display import Image as NotebookImage, display
from PIL import Image as PILImage, ImageDraw
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GRID_SIZE = 8
N_DEPOTS = 6
MAX_STEPS = 120
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ACTION_NAMES = {
    0: "south",
    1: "north",
    2: "east",
    3: "west",
    4: "pickup",
    5: "dropoff",
}

ACTION_DELTAS = {
    0: (1, 0),
    1: (-1, 0),
    2: (0, 1),
    3: (0, -1),
}

DEPOT_NAMES = ["A", "B", "C", "D", "E", "F"]

print("data dir:", DATA_DIR)
print("device:", DEVICE)

data dir: /kaggle/input/competitions/ioai-2026-home-task-2
device: cpu


In [33]:
class DeliverySimulator8x8:
    """Run one 8x8 delivery episode."""

    def reset(self, scenario: dict[str, Any]) -> tuple[int, int, int, int]:
        """Start a scenario and return the compact state."""
        self.step_count = 0
        self.carrying = False
        self.walls = {tuple(cell) for cell in scenario["walls"]}
        self.depots = [tuple(cell) for cell in scenario["depots"]]
        self.agent_pos = tuple(scenario["agent_pos"])
        self.package_location = int(scenario["package_location"])
        self.destination = int(scenario["destination"])
        return self.state()

    def state(self) -> tuple[int, int, int, int]:
        """Return row, column, package field, and destination."""
        package_field = N_DEPOTS if self.carrying else self.package_location
        return int(self.agent_pos[0]), int(self.agent_pos[1]), int(package_field), int(self.destination)

    def can_enter(self, row: int, col: int) -> bool:
        """Check whether the robot can occupy a cell."""
        return 0 <= row < GRID_SIZE and 0 <= col < GRID_SIZE and (row, col) not in self.walls

    def valid_action_mask(self) -> np.ndarray:
        """Return the currently valid actions."""
        row, col, _, destination = self.state()
        mask = np.zeros(6, dtype=bool)
        for action, (dr, dc) in ACTION_DELTAS.items():
            mask[action] = self.can_enter(row + dr, col + dc)
        mask[4] = (not self.carrying) and self.agent_pos == self.depots[self.package_location]
        mask[5] = self.carrying and self.agent_pos == self.depots[destination]
        return mask

    def observation(self) -> dict[str, Any]:
        """Build the model observation for the current state."""
        row, col, package_field, destination = self.state()
        carrying = package_field == N_DEPOTS
        dest_row, dest_col = self.depots[destination]
        target_row, target_col = (dest_row, dest_col) if carrying else self.depots[package_field]

        grid = np.zeros((6, GRID_SIZE, GRID_SIZE), dtype=np.float32)
        for wr, wc in self.walls:
            grid[0, wr, wc] = 1.0
        for dr, dc in self.depots:
            grid[1, dr, dc] = 1.0
        grid[2, row, col] = 1.0
        if not carrying:
            pr, pc = self.depots[package_field]
            grid[3, pr, pc] = 1.0
        grid[4, dest_row, dest_col] = 1.0
        grid[5, :, :] = float(carrying)

        blocked_moves = [float(not self.can_enter(row + dr, col + dc)) for dr, dc in ACTION_DELTAS.values()]
        vector = np.array(
            [
                row / (GRID_SIZE - 1),
                col / (GRID_SIZE - 1),
                package_field / N_DEPOTS,
                destination / (N_DEPOTS - 1),
                float(carrying),
                target_row / (GRID_SIZE - 1),
                target_col / (GRID_SIZE - 1),
                (target_row - row) / (GRID_SIZE - 1),
                (target_col - col) / (GRID_SIZE - 1),
                *blocked_moves,
            ],
            dtype=np.float32,
        )
        return {"grid": grid, "vector": vector, "action_mask": self.valid_action_mask(), "state": self.state()}

    def step(self, action: int) -> tuple[tuple[int, int, int, int], bool, bool, dict[str, Any]]:
        """Apply one action and report episode status."""
        action = int(action)
        done = False
        info = {"invalid_pickup_or_dropoff": False}

        if action in ACTION_DELTAS:
            dr, dc = ACTION_DELTAS[action]
            row, col = self.agent_pos[0] + dr, self.agent_pos[1] + dc
            if self.can_enter(row, col):
                self.agent_pos = (row, col)
        elif action == 4 and (not self.carrying) and self.agent_pos == self.depots[self.package_location]:
            self.carrying = True
        elif action == 5 and self.carrying and self.agent_pos == self.depots[self.destination]:
            done = True
            self.carrying = False
            self.package_location = self.destination
        elif action in (4, 5):
            info["invalid_pickup_or_dropoff"] = True
        else:
            raise ValueError(f"unknown action: {action}")

        self.step_count += 1
        return self.state(), done, self.step_count >= MAX_STEPS and not done, info

    def render(self) -> str:
        """Return an ASCII rendering of the current grid."""
        grid = [["." for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        for row, col in self.walls:
            grid[row][col] = "#"
        for i, (row, col) in enumerate(self.depots):
            grid[row][col] = DEPOT_NAMES[i]

        agent_row, agent_col = self.agent_pos
        grid[agent_row][agent_col] = "T*" if self.carrying else "T"
        rows = [" ".join(f"{cell:>2}" for cell in row) for row in grid]
        package_name = "in taxi" if self.carrying else DEPOT_NAMES[self.package_location]
        rows.append(f"package={package_name}, destination={DEPOT_NAMES[self.destination]}")
        return "\n".join(rows)

In [34]:
with (DATA_DIR / "train_demos.pkl").open("rb") as f:
    train_data = pickle.load(f)
with (DATA_DIR / "valid_scenarios.pkl").open("rb") as f:
    valid_scenarios = pickle.load(f)
with (DATA_DIR / "test_scenarios.pkl").open("rb") as f:
    test_scenarios = pickle.load(f)

train_trajectories = train_data["trajectories"]
steps = [t["num_steps"] for t in train_trajectories]

print("Loaded data")
print("  training demonstrations:", len(train_trajectories))
print("  validation scenarios:", len(valid_scenarios))
print("  test scenarios:", len(test_scenarios))
print("  training state-action samples:", sum(steps))
print("  average demonstration length:", f"{np.mean(steps):.2f}")
print("  expert success rate:", f"{100 * np.mean([t['success'] for t in train_trajectories]):.1f}%")


Loaded data
  training demonstrations: 400
  validation scenarios: 200
  test scenarios: 1600
  training state-action samples: 5327
  average demonstration length: 13.32
  expert success rate: 100.0%


In [35]:
def add_target_channel(obs):
    """
    Adds a 7th channel to the grid.
    The new channel marks the current target cell:
    - before pickup: package location
    - after pickup: destination location
    """

    grid = obs["grid"].astype(np.float32)  # shape: (6, 8, 8)
    vector = obs["vector"].astype(np.float32)

    # In the notebook vector:
    # vector[5] = target_row normalized
    # vector[6] = target_col normalized
    target_row = int(round(vector[5] * (GRID_SIZE - 1)))
    target_col = int(round(vector[6] * (GRID_SIZE - 1)))

    target_channel = np.zeros((1, GRID_SIZE, GRID_SIZE), dtype=np.float32)
    target_channel[0, target_row, target_col] = 1.0

    enhanced_grid = np.concatenate([grid, target_channel], axis=0)

    return enhanced_grid

In [36]:
START_ACTION = 6
PREVIOUS_ACTION_VOCAB = 7


class ModularNavigationDataset(Dataset):
    """
    Mantiene la información del paso anterior de cada trayectoria.

    Cada muestra contiene:
        - cuadrícula actual
        - vector actual
        - acción anterior
        - máscara de acciones válidas
        - acción experta actual
    """

    def __init__(self, trajectories):
        self.samples = []

        for trajectory in trajectories:
            previous_action = START_ACTION

            for obs, action in zip(
                trajectory["observations"],
                trajectory["actions"],
                strict=True
            ):
                self.samples.append({
                    "observation": obs,
                    "previous_action": previous_action,
                    "action": int(action),
                })

                previous_action = int(action)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        obs = sample["observation"]
        previous_action = sample["previous_action"]
        action = sample["action"]

        grid = torch.tensor(
            add_target_channel(obs),
            dtype=torch.float32
        )

        vector = torch.tensor(
            obs["vector"],
            dtype=torch.float32
        )

        previous_action = torch.tensor(
            previous_action,
            dtype=torch.long
        )

        mask = torch.tensor(
            obs["action_mask"],
            dtype=torch.bool
        )

        action = torch.tensor(
            action,
            dtype=torch.long
        )

        return grid, vector, previous_action, mask, action

In [37]:
modular_train_dataset = ModularNavigationDataset(
    train_trajectories
)

modular_train_loader = DataLoader(
    modular_train_dataset,
    batch_size=128,
    shuffle=True
)

grid_example, vector_example, previous_example, mask_example, y_example = (
    modular_train_dataset[0]
)

print("Samples:", len(modular_train_dataset))
print("Grid shape:", grid_example.shape)
print("Vector shape:", vector_example.shape)
print("Previous action:", int(previous_example))
print("Target action:", int(y_example))

Samples: 5327
Grid shape: torch.Size([7, 8, 8])
Vector shape: torch.Size([13])
Previous action: 6
Target action: 1


In [53]:
class ModularNavigationModel(nn.Module):
    """
    Primer módulo:
        CNN que analiza espacialmente la cuadrícula.

    Segundo módulo:
        Cabeza de decisión que combina:
        - característica de la posición del robot
        - característica del objetivo
        - características de las cuatro celdas vecinas
        - resumen global del tablero
        - vector numérico
        - acción anterior
    """

    def __init__(
        self,
        vector_dim=13,
        num_actions=6,
        previous_action_vocab=7
    ):
        super().__init__()

        # No se aplana el mapa aquí.
        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(
                7,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                96,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(96),
            nn.ReLU()
        )

        # Procesamiento de las 13 características globales.
        self.vector_encoder = nn.Sequential(
            nn.Linear(vector_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 64),
            nn.ReLU()
        )

        # Representación aprendida de la acción anterior.
        self.previous_action_embedding = nn.Embedding(
            previous_action_vocab,
            16
        )

        # Se extraen siete grupos espaciales:
        # robot + objetivo + cuatro vecinos + resumen global.
        spatial_decision_dim = 96 * 7

        decision_input_dim = (
            spatial_decision_dim
            + 64   # vector
            + 16   # acción anterior
        )

        self.decision_head = nn.Sequential(
            nn.Linear(decision_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(128, num_actions)
        )

    @staticmethod
    def gather_position_features(
        feature_map,
        rows,
        columns
    ):
        """
        Extrae el vector de características de una posición
        para cada elemento del batch.
        """
        batch_indices = torch.arange(
            feature_map.size(0),
            device=feature_map.device
        )

        return feature_map[
            batch_indices,
            :,
            rows,
            columns
        ]

    def forward(
        self,
        grid,
        vector,
        previous_action
    ):
        """
        grid:
            [batch, 7, 8, 8]

        vector:
            [batch, 13]

        previous_action:
            [batch]
        """

        feature_map = self.spatial_encoder(grid)

        height = feature_map.size(2)
        width = feature_map.size(3)

        # Posición del robot:
        # vector[0] = row normalizada
        # vector[1] = col normalizada
        robot_rows = torch.round(
            vector[:, 0] * (height - 1)
        ).long()

        robot_columns = torch.round(
            vector[:, 1] * (width - 1)
        ).long()

        robot_rows = robot_rows.clamp(
            min=0,
            max=height - 1
        )

        robot_columns = robot_columns.clamp(
            min=0,
            max=width - 1
        )

        # Posición del objetivo:
        # vector[5] = target row normalizada
        # vector[6] = target col normalizada
        target_rows = torch.round(
            vector[:, 5] * (height - 1)
        ).long()

        target_columns = torch.round(
            vector[:, 6] * (width - 1)
        ).long()

        target_rows = target_rows.clamp(
            min=0,
            max=height - 1
        )

        target_columns = target_columns.clamp(
            min=0,
            max=width - 1
        )

        # Características exactamente donde está el robot.
        robot_features = self.gather_position_features(
            feature_map,
            robot_rows,
            robot_columns
        )

        # Características exactamente donde está el objetivo.
        target_features = self.gather_position_features(
            feature_map,
            target_rows,
            target_columns
        )

        # Extraer las cuatro celdas vecinas en el mismo orden
        # de ACTION_DELTAS:
        # south, north, east, west.
        neighbor_features = []

        for action_id in range(4):
            dr, dc = ACTION_DELTAS[action_id]

            neighbor_rows = robot_rows + dr
            neighbor_columns = robot_columns + dc

            valid_position = (
                (neighbor_rows >= 0)
                & (neighbor_rows < height)
                & (neighbor_columns >= 0)
                & (neighbor_columns < width)
            )

            safe_rows = neighbor_rows.clamp(
                min=0,
                max=height - 1
            )

            safe_columns = neighbor_columns.clamp(
                min=0,
                max=width - 1
            )

            current_neighbor_features = (
                self.gather_position_features(
                    feature_map,
                    safe_rows,
                    safe_columns
                )
            )

            # Una posición fuera del tablero se representa con ceros.
            current_neighbor_features = (
                current_neighbor_features
                * valid_position.unsqueeze(1).to(
                    feature_map.dtype
                )
            )

            neighbor_features.append(
                current_neighbor_features
            )

        neighbor_features = torch.cat(
            neighbor_features,
            dim=1
        )

        # Resumen global aprendido del mapa.
        global_features = feature_map.mean(
            dim=(2, 3)
        )

        vector_features = self.vector_encoder(
            vector
        )

        previous_action_features = (
            self.previous_action_embedding(
                previous_action
            )
        )

        decision_features = torch.cat(
            [
                robot_features,
                target_features,
                neighbor_features,
                global_features,
                vector_features,
                previous_action_features
            ],
            dim=1
        )

        return self.decision_head(
            decision_features
        )

In [54]:
modular_model = ModularNavigationModel(
    vector_dim=vector_example.numel(),
    num_actions=6
).to(DEVICE)

print(modular_model)

total_parameters = sum(
    parameter.numel()
    for parameter in modular_model.parameters()
)

print("Parameters:", total_parameters)

ModularNavigationModel(
  (spatial_encoder): Sequential(
    (0): Conv2d(7, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
  )
  (vector_encoder): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): R

In [55]:
modular_action_counts = Counter(
    sample["action"]
    for sample in modular_train_dataset.samples
)

modular_counts = torch.zeros(
    6,
    dtype=torch.float32
)

for action_id, count in modular_action_counts.items():
    modular_counts[action_id] = count

modular_class_weights = (
    modular_counts.sum()
    / (
        6
        * modular_counts.clamp(min=1)
    )
)

modular_class_weights = (
    modular_class_weights
    / modular_class_weights.mean()
)

modular_class_weights = modular_class_weights.to(
    DEVICE
)

print("Action counts:", modular_action_counts)
print("Class weights:", modular_class_weights)

Action counts: Counter({3: 1193, 1: 1148, 2: 1120, 0: 1066, 4: 400, 5: 400})
Class weights: tensor([0.6591, 0.6120, 0.6273, 0.5889, 1.7564, 1.7564])


In [56]:
modular_optimizer = torch.optim.AdamW(
    modular_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

modular_criterion = nn.CrossEntropyLoss(
    weight=modular_class_weights
)

In [57]:
@torch.no_grad()
def modular_train_accuracy():
    modular_model.eval()

    correct = 0
    total = 0

    accuracy_loader = DataLoader(
        modular_train_dataset,
        batch_size=1024,
        shuffle=False
    )

    for (
        grid,
        vector,
        previous_action,
        mask,
        y
    ) in accuracy_loader:

        grid = grid.to(DEVICE)
        vector = vector.to(DEVICE)
        previous_action = previous_action.to(DEVICE)
        mask = mask.to(DEVICE)
        y = y.to(DEVICE)

        logits = modular_model(
            grid,
            vector,
            previous_action
        )

        logits = logits.masked_fill(
            ~mask,
            -1e9
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += int(
            (predictions == y).sum().item()
        )

        total += y.size(0)

    return correct / total

In [58]:
MODULAR_EPOCHS = 30

for epoch in tqdm(
    range(1, MODULAR_EPOCHS + 1)
):
    modular_model.train()

    total_loss = 0.0
    total_examples = 0

    for (
        grid,
        vector,
        previous_action,
        mask,
        y
    ) in modular_train_loader:

        grid = grid.to(DEVICE)
        vector = vector.to(DEVICE)
        previous_action = previous_action.to(DEVICE)
        mask = mask.to(DEVICE)
        y = y.to(DEVICE)

        logits = modular_model(
            grid,
            vector,
            previous_action
        )

        logits = logits.masked_fill(
            ~mask,
            -1e9
        )

        loss = modular_criterion(
            logits,
            y
        )

        modular_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            modular_model.parameters(),
            max_norm=5.0
        )

        modular_optimizer.step()

        batch_size = y.size(0)

        total_loss += (
            float(loss.item())
            * batch_size
        )

        total_examples += batch_size

    average_loss = (
        total_loss
        / total_examples
    )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == MODULAR_EPOCHS
    ):
        train_accuracy = modular_train_accuracy()

        print(
            f"epoch {epoch:02d} | "
            f"loss {average_loss:.4f} | "
            f"train action acc {train_accuracy:.3f}"
        )

  0%|          | 0/30 [00:00<?, ?it/s]

epoch 01 | loss 0.4346 | train action acc 0.948
epoch 05 | loss 0.0185 | train action acc 0.998
epoch 10 | loss 0.0041 | train action acc 1.000
epoch 15 | loss 0.0002 | train action acc 1.000
epoch 20 | loss 0.0244 | train action acc 0.998
epoch 25 | loss 0.0077 | train action acc 0.998
epoch 30 | loss 0.0047 | train action acc 1.000


In [59]:
@torch.no_grad()
def modular_model_action(
    obs,
    previous_action
):
    modular_model.eval()

    grid = torch.tensor(
        add_target_channel(obs),
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    vector = torch.tensor(
        obs["vector"],
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    previous_action_tensor = torch.tensor(
        [previous_action],
        dtype=torch.long,
        device=DEVICE
    )

    mask = torch.tensor(
        obs["action_mask"],
        dtype=torch.bool,
        device=DEVICE
    )

    logits = modular_model(
        grid,
        vector,
        previous_action_tensor
    ).squeeze(0)

    logits = logits.masked_fill(
        ~mask,
        -1e9
    )

    return int(
        logits.argmax().item()
    )
def run_episode_modular(
    scenario,
    max_steps=MAX_STEPS,
    render=False
):
    simulator = DeliverySimulator8x8()
    simulator.reset(scenario)

    previous_action = START_ACTION

    frames = []
    actions = []

    invalid_pickup_or_dropoff = 0
    done = False

    if render:
        frames.append(
            simulator.render()
        )

    for _ in range(max_steps):
        obs = simulator.observation()

        action = modular_model_action(
            obs,
            previous_action
        )

        _, done, timed_out, info = (
            simulator.step(action)
        )

        actions.append(action)

        invalid_pickup_or_dropoff += int(
            info["invalid_pickup_or_dropoff"]
        )

        previous_action = action

        if render:
            frames.append(
                simulator.render()
            )

        if done or timed_out:
            break

    return {
        "success": done,
        "steps": len(actions),
        "actions": actions,
        "frames": frames,
        "invalid_pickup_or_dropoff": (
            invalid_pickup_or_dropoff
        ),
    }
def evaluate_modular_model(scenarios):
    results = [
        run_episode_modular(scenario)
        for scenario in tqdm(
            scenarios,
            desc="Evaluating modular navigation model"
        )
    ]

    return {
        "success_rate": float(
            np.mean([
                result["success"]
                for result in results
            ])
        ),
        "avg_steps": float(
            np.mean([
                result["steps"]
                for result in results
            ])
        ),
        "avg_invalid_pickup_or_dropoff": float(
            np.mean([
                result["invalid_pickup_or_dropoff"]
                for result in results
            ])
        ),
        "results": results,
    }

In [60]:
modular_eval = evaluate_modular_model(valid_scenarios)

print("Modular model:", modular_eval["success_rate"])
print("Average steps:", modular_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    modular_eval["avg_invalid_pickup_or_dropoff"]
)


Evaluating modular navigation model:   0%|          | 0/200 [00:00<?, ?it/s]

Modular model: 0.895
Average steps: 24.42
Invalid pickup/dropoff: 0.0


In [61]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)

GRU_USE_ROTATIONS = True

GRU_DELTA_TO_ACTION = {
    delta: action
    for action, delta in ACTION_DELTAS.items()
}


def gru_rotate_coordinate(row, col, rotations):
    """Rotate one coordinate 90 degrees counterclockwise `rotations` times."""
    row = int(row)
    col = int(col)

    for _ in range(rotations % 4):
        row, col = GRID_SIZE - 1 - col, row

    return row, col


def gru_rotate_delta(dr, dc, rotations):
    """Rotate one movement delta 90 degrees counterclockwise."""
    dr = int(dr)
    dc = int(dc)

    for _ in range(rotations % 4):
        dr, dc = -dc, dr

    return dr, dc


def gru_rotate_action(action, rotations):
    """
    Rotate a movement action.

    pickup, dropoff and START_ACTION do not change.
    """
    action = int(action)

    if action not in ACTION_DELTAS:
        return action

    dr, dc = ACTION_DELTAS[action]
    rotated_delta = gru_rotate_delta(
        dr,
        dc,
        rotations,
    )

    return GRU_DELTA_TO_ACTION[rotated_delta]


def gru_rotate_observation(obs, rotations):
    """
    Rotate every spatial component of an observation consistently.

    The same transformation is applied to:
    - grid channels;
    - agent coordinates;
    - current-target coordinates;
    - relative target direction;
    - blocked-direction features;
    - action mask.
    """
    rotations %= 4

    rotated_grid = np.rot90(
        obs["grid"],
        k=rotations,
        axes=(1, 2),
    ).copy()

    old_vector = np.asarray(
        obs["vector"],
        dtype=np.float32,
    )
    new_vector = old_vector.copy()

    row, col, package_field, destination = obs["state"]

    new_row, new_col = gru_rotate_coordinate(
        row,
        col,
        rotations,
    )

    target_row = int(round(
        float(old_vector[5]) * (GRID_SIZE - 1)
    ))
    target_col = int(round(
        float(old_vector[6]) * (GRID_SIZE - 1)
    ))

    new_target_row, new_target_col = gru_rotate_coordinate(
        target_row,
        target_col,
        rotations,
    )

    new_vector[0] = new_row / (GRID_SIZE - 1)
    new_vector[1] = new_col / (GRID_SIZE - 1)

    new_vector[2] = old_vector[2]
    new_vector[3] = old_vector[3]
    new_vector[4] = old_vector[4]

    new_vector[5] = new_target_row / (GRID_SIZE - 1)
    new_vector[6] = new_target_col / (GRID_SIZE - 1)
    new_vector[7] = (
        new_target_row - new_row
    ) / (GRID_SIZE - 1)
    new_vector[8] = (
        new_target_col - new_col
    ) / (GRID_SIZE - 1)

    rotated_blocked = np.zeros(
        4,
        dtype=np.float32,
    )

    for old_action in range(4):
        new_action = gru_rotate_action(
            old_action,
            rotations,
        )
        rotated_blocked[new_action] = old_vector[9 + old_action]

    new_vector[9:13] = rotated_blocked

    old_mask = np.asarray(
        obs["action_mask"],
        dtype=bool,
    )
    new_mask = np.zeros_like(old_mask)

    for old_action in range(6):
        new_action = gru_rotate_action(
            old_action,
            rotations,
        )
        new_mask[new_action] = old_mask[old_action]

    return {
        "grid": rotated_grid,
        "vector": new_vector.astype(np.float32),
        "action_mask": new_mask,
        "state": (
            new_row,
            new_col,
            int(package_field),
            int(destination),
        ),
    }


In [62]:
class SequentialNavigationDataset(Dataset):
    """
    Returns complete expert trajectories instead of independent steps.

    One rotation, when enabled, is applied consistently to every step
    of the same trajectory.
    """

    def __init__(
        self,
        trajectories,
        use_rotations=False,
    ):
        self.items = []
        self.action_counts = Counter()

        rotations = (
            [0, 1, 2, 3]
            if use_rotations
            else [0]
        )

        for trajectory in trajectories:
            for rotation in rotations:
                self.items.append(
                    (trajectory, rotation)
                )

                for action in trajectory["actions"]:
                    rotated_action = gru_rotate_action(
                        int(action),
                        rotation,
                    )
                    self.action_counts[rotated_action] += 1

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        trajectory, rotation = self.items[index]

        grids = []
        vectors = []
        previous_actions = []
        masks = []
        actions = []
        remaining_budgets = []

        previous_action = START_ACTION

        for step_index, (obs, action) in enumerate(zip(
            trajectory["observations"],
            trajectory["actions"],
            strict=True,
        )):
            rotated_obs = gru_rotate_observation(
                obs,
                rotation,
            )
            rotated_action = gru_rotate_action(
                int(action),
                rotation,
            )
            rotated_previous_action = gru_rotate_action(
                previous_action,
                rotation,
            )

            grids.append(torch.tensor(
                add_target_channel(rotated_obs),
                dtype=torch.float32,
            ))
            vectors.append(torch.tensor(
                rotated_obs["vector"],
                dtype=torch.float32,
            ))
            previous_actions.append(
                rotated_previous_action
            )
            masks.append(torch.tensor(
                rotated_obs["action_mask"],
                dtype=torch.bool,
            ))
            actions.append(rotated_action)

            remaining_budget = max(
                0.0,
                (MAX_STEPS - step_index) / MAX_STEPS,
            )
            remaining_budgets.append([
                remaining_budget
            ])

            previous_action = int(action)

        return {
            "grids": torch.stack(grids),
            "vectors": torch.stack(vectors),
            "previous_actions": torch.tensor(
                previous_actions,
                dtype=torch.long,
            ),
            "masks": torch.stack(masks),
            "actions": torch.tensor(
                actions,
                dtype=torch.long,
            ),
            "remaining_budgets": torch.tensor(
                remaining_budgets,
                dtype=torch.float32,
            ),
        }


def collate_navigation_sequences(batch):
    """Pad variable-length trajectories while preserving their lengths."""
    lengths = torch.tensor(
        [item["actions"].size(0) for item in batch],
        dtype=torch.long,
    )

    return {
        "grids": pad_sequence(
            [item["grids"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "vectors": pad_sequence(
            [item["vectors"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "previous_actions": pad_sequence(
            [item["previous_actions"] for item in batch],
            batch_first=True,
            padding_value=START_ACTION,
        ),
        "masks": pad_sequence(
            [item["masks"] for item in batch],
            batch_first=True,
            padding_value=False,
        ),
        "actions": pad_sequence(
            [item["actions"] for item in batch],
            batch_first=True,
            padding_value=-100,
        ),
        "remaining_budgets": pad_sequence(
            [item["remaining_budgets"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "lengths": lengths,
    }


In [63]:
gru_train_dataset = SequentialNavigationDataset(
    train_trajectories,
    use_rotations=GRU_USE_ROTATIONS,
)

gru_train_loader = DataLoader(
    gru_train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_navigation_sequences,
)

gru_example = gru_train_dataset[0]

print("Sequential trajectories:", len(gru_train_dataset))
print("Example length:", gru_example["actions"].size(0))
print("Grid sequence shape:", gru_example["grids"].shape)
print("Vector sequence shape:", gru_example["vectors"].shape)
print("Remaining-budget shape:", gru_example["remaining_budgets"].shape)
print("Rotations enabled:", GRU_USE_ROTATIONS)
print("Action counts:", gru_train_dataset.action_counts)


Sequential trajectories: 1600
Example length: 23
Grid sequence shape: torch.Size([23, 7, 8, 8])
Vector sequence shape: torch.Size([23, 13])
Remaining-budget shape: torch.Size([23, 1])
Rotations enabled: True
Action counts: Counter({1: 4527, 2: 4527, 0: 4527, 3: 4527, 4: 1600, 5: 1600})


In [67]:
class CNNGRUNavigationModel(nn.Module):
    """
    CNN spatial encoder followed by one GRU and one action head.

    The GRU hidden state acts as learned temporal memory. The model receives
    complete trajectories during training and one step at a time during
    inference.
    """

    def __init__(
        self,
        vector_dim=13,
        num_actions=6,
        previous_action_vocab=7,
        hidden_dim=192,
    ):
        super().__init__()

        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(
                7,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 32),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                96,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 96),
            nn.ReLU(),
        )

        self.vector_encoder = nn.Sequential(
            nn.Linear(vector_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 64),
            nn.ReLU(),
        )

        self.previous_action_embedding = nn.Embedding(
            previous_action_vocab,
            16,
        )
        self.full_map_encoder = nn.Sequential(
            nn.Conv2d(
                96,
                16,
                kernel_size=1,
            ),
            nn.ReLU(),
            nn.Flatten(),
        )
        local_spatial_dim = 96 * 7
        full_map_dim = 16 * GRID_SIZE * GRID_SIZE
        
        spatial_decision_dim = (
            local_spatial_dim
            + full_map_dim
        )

        combined_dim = (
            spatial_decision_dim
            + 64
            + 16
            + 1
        )

        self.input_projection = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256, hidden_dim),
            nn.ReLU(),
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )

        self.output_dropout = nn.Dropout(0.15)
        self.action_head = nn.Linear(
            hidden_dim,
            num_actions,
        )
    @staticmethod
    def gather_position_features(
        feature_map,
        rows,
        columns,
    ):
        batch_indices = torch.arange(
            feature_map.size(0),
            device=feature_map.device,
        )

        return feature_map[
            batch_indices,
            :,
            rows,
            columns,
        ]

    def encode_steps(
        self,
        grids,
        vectors,
        previous_actions,
        remaining_budgets,
    ):
        feature_map = self.spatial_encoder(
            grids
        )

        height = feature_map.size(2)
        width = feature_map.size(3)

        robot_rows = torch.round(
            vectors[:, 0] * (height - 1)
        ).long().clamp(
            min=0,
            max=height - 1,
        )

        robot_columns = torch.round(
            vectors[:, 1] * (width - 1)
        ).long().clamp(
            min=0,
            max=width - 1,
        )

        target_rows = torch.round(
            vectors[:, 5] * (height - 1)
        ).long().clamp(
            min=0,
            max=height - 1,
        )

        target_columns = torch.round(
            vectors[:, 6] * (width - 1)
        ).long().clamp(
            min=0,
            max=width - 1,
        )

        robot_features = self.gather_position_features(
            feature_map,
            robot_rows,
            robot_columns,
        )

        target_features = self.gather_position_features(
            feature_map,
            target_rows,
            target_columns,
        )

        neighbor_features = []

        for action_id in range(4):
            dr, dc = ACTION_DELTAS[action_id]

            neighbor_rows = robot_rows + dr
            neighbor_columns = robot_columns + dc

            valid_position = (
                (neighbor_rows >= 0)
                & (neighbor_rows < height)
                & (neighbor_columns >= 0)
                & (neighbor_columns < width)
            )

            safe_rows = neighbor_rows.clamp(
                min=0,
                max=height - 1,
            )
            safe_columns = neighbor_columns.clamp(
                min=0,
                max=width - 1,
            )

            current_features = self.gather_position_features(
                feature_map,
                safe_rows,
                safe_columns,
            )

            current_features = (
                current_features
                * valid_position.unsqueeze(1).to(
                    feature_map.dtype
                )
            )

            neighbor_features.append(
                current_features
            )

        neighbor_features = torch.cat(
            neighbor_features,
            dim=1,
        )

        global_features = feature_map.mean(
            dim=(2, 3)
        )
        full_map_features = self.full_map_encoder(
            feature_map
        )
        vector_features = self.vector_encoder(
            vectors
        )

        previous_action_features = (
            self.previous_action_embedding(
                previous_actions
            )
        )

        combined = torch.cat(
            [
                robot_features,
                target_features,
                neighbor_features,
                global_features,
                full_map_features,
                vector_features,
                previous_action_features,
                remaining_budgets,
            ],
            dim=1,
        )
        

        return self.input_projection(
            combined
        )

    def forward(
        self,
        grids,
        vectors,
        previous_actions,
        remaining_budgets,
        lengths=None,
        hidden=None,
    ):
        batch_size, sequence_length = grids.shape[:2]

        flat_step_features = self.encode_steps(
            grids.reshape(
                batch_size * sequence_length,
                *grids.shape[2:],
            ),
            vectors.reshape(
                batch_size * sequence_length,
                vectors.size(-1),
            ),
            previous_actions.reshape(-1),
            remaining_budgets.reshape(-1, 1),
        )

        sequence_features = flat_step_features.reshape(
            batch_size,
            sequence_length,
            -1,
        )

        if lengths is not None:
            packed = pack_padded_sequence(
                sequence_features,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False,
            )

            packed_output, new_hidden = self.gru(
                packed,
                hidden,
            )

            recurrent_output, _ = pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=sequence_length,
            )
        else:
            recurrent_output, new_hidden = self.gru(
                sequence_features,
                hidden,
            )

        logits = self.action_head(
            self.output_dropout(
                recurrent_output
            )
        )

        return logits, new_hidden


In [68]:
gru_model = CNNGRUNavigationModel(
    vector_dim=gru_example["vectors"].size(-1),
    num_actions=6,
    hidden_dim=192,
).to(DEVICE)

gru_counts = torch.zeros(
    6,
    dtype=torch.float32,
)

for action_id, count in gru_train_dataset.action_counts.items():
    gru_counts[action_id] = count

gru_class_weights = (
    gru_counts.sum()
    / (
        6
        * gru_counts.clamp(min=1)
    )
)
gru_class_weights = (
    gru_class_weights
    / gru_class_weights.mean()
).to(DEVICE)

gru_optimizer = torch.optim.AdamW(
    gru_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4,
)

gru_criterion = nn.CrossEntropyLoss(
    weight=gru_class_weights,
    ignore_index=-100,
)

print(gru_model)
print(
    "GRU parameters:",
    sum(
        parameter.numel()
        for parameter in gru_model.parameters()
    ),
)
print("GRU class weights:", gru_class_weights)


CNNGRUNavigationModel(
  (spatial_encoder): Sequential(
    (0): Conv2d(7, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): GroupNorm(8, 64, eps=1e-05, affine=True)
    (5): ReLU()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): GroupNorm(8, 64, eps=1e-05, affine=True)
    (8): ReLU()
    (9): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): GroupNorm(8, 96, eps=1e-05, affine=True)
    (11): ReLU()
  )
  (vector_encoder): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (previous_action_embedding): Embedding(7, 16)
  (full_map_encoder): Sequential(
    (0): Conv2d(96, 16, kernel_size=(1, 1), stride=(1, 1))
    (1

In [70]:
@torch.no_grad()
def gru_train_accuracy():
    gru_model.eval()

    correct = 0
    total = 0

    accuracy_loader = DataLoader(
        gru_train_dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_navigation_sequences,
    )

    for batch in accuracy_loader:
        grids = batch["grids"].to(DEVICE)
        vectors = batch["vectors"].to(DEVICE)
        previous_actions = batch["previous_actions"].to(DEVICE)
        masks = batch["masks"].to(DEVICE)
        actions = batch["actions"].to(DEVICE)
        remaining_budgets = batch["remaining_budgets"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)

        logits, _ = gru_model(
            grids,
            vectors,
            previous_actions,
            remaining_budgets,
            lengths=lengths,
        )

        masked_logits = logits.masked_fill(
            ~masks,
            -1e9,
        )

        predictions = masked_logits.argmax(
            dim=-1
        )

        valid_steps = actions != -100

        correct += int(
            (
                (predictions == actions)
                & valid_steps
            ).sum().item()
        )
        total += int(
            valid_steps.sum().item()
        )

    return correct / max(total, 1)


In [71]:
GRU_EPOCHS = 25

for epoch in tqdm(
    range(1, GRU_EPOCHS + 1),
    desc="Training CNN + GRU",
):
    gru_model.train()

    total_loss = 0.0
    total_valid_steps = 0

    for batch in gru_train_loader:
        grids = batch["grids"].to(DEVICE)
        vectors = batch["vectors"].to(DEVICE)
        previous_actions = batch["previous_actions"].to(DEVICE)
        masks = batch["masks"].to(DEVICE)
        actions = batch["actions"].to(DEVICE)
        remaining_budgets = batch["remaining_budgets"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)

        logits, _ = gru_model(
            grids,
            vectors,
            previous_actions,
            remaining_budgets,
            lengths=lengths,
        )

        masked_logits = logits.masked_fill(
            ~masks,
            -1e9,
        )

        loss = gru_criterion(
            masked_logits.reshape(-1, 6),
            actions.reshape(-1),
        )

        gru_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gru_model.parameters(),
            max_norm=2.0,
        )

        gru_optimizer.step()

        valid_steps = int(
            (actions != -100).sum().item()
        )

        total_loss += (
            float(loss.item())
            * valid_steps
        )
        total_valid_steps += valid_steps

    average_loss = (
        total_loss
        / max(total_valid_steps, 1)
    )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == GRU_EPOCHS
    ):
        train_accuracy = gru_train_accuracy()

        print(
            f"epoch {epoch:02d} | "
            f"loss {average_loss:.4f} | "
            f"train action acc {train_accuracy:.3f}"
        )


Training CNN + GRU:   0%|          | 0/25 [00:00<?, ?it/s]

epoch 01 | loss 0.5226 | train action acc 0.871
epoch 05 | loss 0.1586 | train action acc 0.932
epoch 10 | loss 0.0617 | train action acc 0.975
epoch 15 | loss 0.0260 | train action acc 0.994
epoch 20 | loss 0.0126 | train action acc 0.996
epoch 25 | loss 0.0119 | train action acc 0.997


In [72]:
@torch.no_grad()
def gru_model_action(
    obs,
    previous_action,
    remaining_budget,
    hidden,
):
    """
    Predict one action and return the updated GRU memory.

    Only the environment action mask is applied.
    """
    gru_model.eval()

    grid = torch.tensor(
        add_target_channel(obs),
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    vector = torch.tensor(
        obs["vector"],
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    previous_action_tensor = torch.tensor(
        [[previous_action]],
        dtype=torch.long,
        device=DEVICE,
    )

    remaining_budget_tensor = torch.tensor(
        [[[remaining_budget]]],
        dtype=torch.float32,
        device=DEVICE,
    )

    logits, new_hidden = gru_model(
        grid,
        vector,
        previous_action_tensor,
        remaining_budget_tensor,
        hidden=hidden,
    )

    logits = logits[0, 0]

    mask = torch.tensor(
        obs["action_mask"],
        dtype=torch.bool,
        device=DEVICE,
    )

    logits = logits.masked_fill(
        ~mask,
        -1e9,
    )

    action = int(
        logits.argmax().item()
    )

    return action, new_hidden


def run_episode_gru(
    scenario,
    max_steps=MAX_STEPS,
    render=False,
):
    simulator = DeliverySimulator8x8()
    simulator.reset(scenario)

    previous_action = START_ACTION
    hidden = None

    frames = []
    actions = []

    invalid_pickup_or_dropoff = 0
    done = False

    if render:
        frames.append(
            simulator.render()
        )

    for step_index in range(max_steps):
        obs = simulator.observation()

        remaining_budget = max(
            0.0,
            (max_steps - step_index) / max_steps,
        )

        action, hidden = gru_model_action(
            obs,
            previous_action,
            remaining_budget,
            hidden,
        )

        _, done, timed_out, info = (
            simulator.step(action)
        )

        actions.append(action)

        invalid_pickup_or_dropoff += int(
            info["invalid_pickup_or_dropoff"]
        )

        previous_action = action

        if render:
            frames.append(
                simulator.render()
            )

        if done or timed_out:
            break

    return {
        "success": done,
        "steps": len(actions),
        "actions": actions,
        "frames": frames,
        "invalid_pickup_or_dropoff": (
            invalid_pickup_or_dropoff
        ),
    }


def evaluate_gru_model(scenarios):
    results = [
        run_episode_gru(scenario)
        for scenario in tqdm(
            scenarios,
            desc="Evaluating CNN + GRU",
        )
    ]

    return {
        "success_rate": float(
            np.mean([
                result["success"]
                for result in results
            ])
        ),
        "avg_steps": float(
            np.mean([
                result["steps"]
                for result in results
            ])
        ),
        "avg_invalid_pickup_or_dropoff": float(
            np.mean([
                result["invalid_pickup_or_dropoff"]
                for result in results
            ])
        ),
        "results": results,
    }


In [73]:
gru_eval = evaluate_gru_model(
    valid_scenarios
)

print("CNN + GRU:", gru_eval["success_rate"])
print("Average steps:", gru_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    gru_eval["avg_invalid_pickup_or_dropoff"],
)

print()
print("Baseline modular:", modular_eval["success_rate"])
print("CNN + GRU:", gru_eval["success_rate"])


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

CNN + GRU: 0.94
Average steps: 19.865
Invalid pickup/dropoff: 0.0

Baseline modular: 0.895
CNN + GRU: 0.94


In [74]:
successful_episodes = sum(
    result["success"]
    for result in gru_eval["results"]
)

total_episodes = len(gru_eval["results"])

score = successful_episodes / total_episodes

print(f"Successful episodes: {successful_episodes}/{total_episodes}")
print(f"Score: {score:.4f}")
print(f"Score percentage: {score * 100:.2f}%")

Successful episodes: 188/200
Score: 0.9400
Score percentage: 94.00%


In [75]:
modular_eval = evaluate_modular_model(valid_scenarios)

print("Score:", modular_eval["success_rate"])
print("Average steps:", modular_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    modular_eval["avg_invalid_pickup_or_dropoff"]
)

Evaluating modular navigation model:   0%|          | 0/200 [00:00<?, ?it/s]

Score: 0.895
Average steps: 24.42
Invalid pickup/dropoff: 0.0


In [76]:
successful_episodes = sum(
    result["success"]
    for result in modular_eval["results"]
)

total_episodes = len(modular_eval["results"])

score = successful_episodes / total_episodes

print(f"Successful episodes: {successful_episodes}/{total_episodes}")
print(f"Score: {score:.4f}")
print(f"Score percentage: {score * 100:.2f}%")

Successful episodes: 179/200
Score: 0.8950
Score percentage: 89.50%


In [77]:
import json
import pandas as pd
from tqdm.auto import tqdm

submission_rows = []

for scenario in tqdm(
    test_scenarios,
    desc="Generating modular submission",
):
    episode = run_episode_gru(scenario)

    action_ids = [
        int(action)
        for action in episode["actions"]
    ]

    submission_rows.append({
        "id": (
            f'{scenario["layout_id"]}'
            f'__{scenario["episode_seed"]}'
        ),
        "actions": json.dumps(
            action_ids,
            separators=(",", ":"),
        ),
    })

submission = pd.DataFrame(
    submission_rows,
    columns=["id", "actions"],
)

# Verificaciones antes de guardar
assert len(submission) == len(test_scenarios)
assert submission.columns.tolist() == ["id", "actions"]
assert submission["id"].is_unique
assert not submission.isna().any().any()

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(
    submission_path,
    index=False,
)

print("Saved:", submission_path)
print("Shape:", submission.shape)
print(submission.head())


Generating modular submission:   0%|          | 0/1600 [00:00<?, ?it/s]

Saved: /kaggle/working/submission.csv
Shape: (1600, 2)
                  id                                      actions
0  test_0000__300000                        [1,2,2,0,4,1,3,3,1,5]
1  test_0000__300001              [2,2,2,1,4,3,3,3,3,3,1,3,3,0,5]
2  test_0000__300002                [0,0,4,0,0,2,2,2,0,0,0,2,2,5]
3  test_0000__300003  [3,3,3,3,1,1,3,1,1,1,1,3,3,0,4,0,0,0,0,0,5]
4  test_0001__300004                                  [3,0,4,1,5]
